## 依赖导入

本例集合覆盖概率论与数理统计的常用计算：分布的密度/概率、数字特征、描述性统计、QQ 图与直方图、置信区间、假设检验（Z 检验、单样本/双样本 t 检验、卡方拟合优度、K-S 正态性检验）以及单因素/双因素方差分析。统一从 `scipy.stats`、`statsmodels` 引入。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as ss
import statsmodels.api as sm
from scipy.optimize import fsolve
from scipy.stats import binom, expon, f, gamma, norm, probplot, sem, t
from statsmodels.stats.weightstats import ttest_ind, ztest

## 全局绘图配置

设置全局 `plt.rc`：启用 TeX 渲染数学公式、中文字体 SimHei、字号 16、关闭坐标轴负号的Unicode 显示，保证后续各图的中文与公式标签正常显示。

In [ ]:
plt.rc("text", usetex=True)
plt.rc("font", size=15)
plt.rc("font", family="SimHei")
plt.rc("font", size=16)
plt.rc("axes", unicode_minus=False)

## 7.1  Gamma 与指数分布的概率密度

在同一画布并排绘制尺度参数 $\theta$ 取 $\frac{1}{3},1,2$ 时的 Gamma(1, $\theta$) 与 Exponential($\theta$) 概率密度曲线（源程序 ex9_1.py）。两个子图各用三种线型区分三种尺度。

In [ ]:
x = np.linspace(0, 3, 100)
scales = [1 / 3, 1, 2]
styles = ["*-", ".-", "o-"]
labels = [r"$\theta=\frac{1}{3}$", r"$\theta=1$", r"$\theta=2$"]

fig, (ax, ax2) = plt.subplots(1, 2)
for sc, st, lb in zip(scales, styles, labels):
    ax.plot(x, gamma.pdf(x, 1, scale=sc), st, label=lb)
    ax2.plot(x, expon.pdf(x, scale=sc), st, label=lb)
for a in (ax, ax2):
    a.set(xlabel="$x$", ylabel="$f(x)$")
    a.legend()
plt.show()

## 7.2  二项分布的概率质量函数

计算 $B(6,\,0.3)$ 各取值的概率质量并绘图（源程序 ex9_2.py）。左子图用红色圆点标出概率、并以竖线（`vlines`，线宽 `lw=2`、透明度 `alpha=0.5`）连接横轴；右子图用火柴图 `stem` 呈现同一分布，结果保存为 `data/9_2.png`。

In [ ]:
n, p = 6, 0.3
x = np.arange(7)
y = binom.pmf(x, n, p)

fig, (ax, ax2) = plt.subplots(1, 2)
ax.plot(x, y, "ro")
ax.vlines(x, 0, y, "k", lw=2, alpha=0.5)
ax2.stem(x, y, use_line_collection=True)
fig.savefig("data/9_2.png", dpi=500)
plt.show()

## 7.3  正态分布分位数与方程求根

先求 $N(3,2^2)$ 的 0.25 分位数 `c1`；再以 `fsolve` 从初值 1 出发求 $1-\Phi(c)-3\Phi(c)=0$ 的零点 `c2`（源程序 ex9_3.py）。

In [ ]:
c1 = norm.ppf(0.25, 3, 2)

def fc(c):
    return 1 - norm.cdf(c, 3, 2) - 3 * norm.cdf(c, 3, 2)

c2 = fsolve(fc, 1)
print("c1=", c1)
print("c2=", c2)

## 7.4  指数分布的数字特征

用 `expon.stats` 一次性返回尺度参数为 3 的指数分布的均值、方差、偏度、峰度（mvsk，源程序 ex9_4.py）。

In [ ]:
print(expon.stats(scale=3, moments="mvsk"))

## 7.5  按行计算多种描述统计量

读取 `data/9_5.txt`（无表头，行表示甲、乙两组成绩）转为数组后，按行（`axis=1`）分别计算均值、中位数、极差、样本方差（`ddof=1`）、样本标准差，并对整个矩阵计算协方差矩阵与相关系数矩阵（源程序 ex9_5.py）。

In [ ]:
a = pd.read_csv("data/9_5.txt", header=None)
b = a.values
mu = np.mean(b, axis=1)
zw = np.median(b, axis=1)
jc = np.ptp(b, axis=1)
fc = np.var(b, axis=1, ddof=1)
bz = np.std(b, axis=1, ddof=1)
xf = np.cov(b)
xs = np.corrcoef(b)

## 7.6  转置后用 pandas 汇总描述统计

读取同一数据并转置（列变组、行为样本），用 `describe` 输出计数/均值/标准差/分位数等，再分别输出偏度 `skew`、峰度 `kurt` 与 90% 分位数 `quantile(0.9)`（源程序 ex9_6.py）。

In [ ]:
df = pd.read_csv("data/9_5.txt", header=None).T
print(df.describe())
print("-----\n偏度：\n", df.skew())
print("-----\n峰度：\n", df.kurt())
print("-----\n90% 分位数：\n", df.quantile(0.9))

## 7.7  直方图的两种画法

转置数据后用 `ax.hist` 分列绘制两组的 5 桶直方图并打印第一组的分组结果；随后用 `df.hist(bins=5)` 演示 pandas 自带的分面直方图（另一种画法，源程序 ex9_7.py）。

In [ ]:
df = pd.read_csv("data/9_5.txt", header=None).T
fig, (ax, ax2) = plt.subplots(1, 2)
h1 = ax.hist(df[0], 5)
print(h1)
ax2.hist(df[1], 5)
df.hist(bins=5)
plt.show()

## 7.8  箱线图对比两组

转置数据后用 `ax.boxplot` 绘制甲、乙两班成绩的箱线图，横轴标签为“甲班”“乙班”（源程序 ex9_8.py）。

In [ ]:
df = pd.read_csv("data/9_5.txt", header=None).T
fig, ax = plt.subplots()
ax.boxplot(df, labels=["甲班", "乙班"])
plt.show()

## 7.9  累积频率直方图

提取甲班成绩（转置后第 0 列），以 `density=True, histtype="step", cumulative=True` 绘制累积分布阶梯直方图并打印分组结果，加网格（源程序 ex9_9.py）。

In [ ]:
df = pd.read_csv("data/9_5.txt", header=None)
d = df.T[0]
fig, ax = plt.subplots()
h = ax.hist(d, density=True, histtype="step", cumulative=True)
print(h)
ax.grid()
plt.show()

## 7.10  QQ 图与正态概率图

提取甲班成绩，排序后用理论正态分位数作横轴、样本排序值作纵轴画 QQ 图，并叠加对角参照直线判断正态性；右子图用 `probplot` 直接绘制正态概率图（源程序 ex9_10.py）。

In [ ]:
df = pd.read_csv("data/9_5.txt", header=None)
d = df.values[0]
mu, s = d.mean(), d.std()
sd = np.sort(d)
n = len(d)
x = (np.arange(n) + 0.5) / n
yi = norm.ppf(x, mu, s)

fig, (ax, ax2) = plt.subplots(1, 2)
ax.plot(yi, sd, "o", label="QQ 图")
ax.plot(sd, sd, label="参照直线")
ax.legend()
probplot(d, plot=plt)
plt.show()

## 7.11  均值的 t 置信区间

读取 `data/9_11.txt` 展平，计算样本均值与标准误 `sem`，用自由度 $n-1$ 的双侧 95% t 分位数构造均值的置信区间并保留 4 位小数（源程序 ex9_11.py）。

In [ ]:
d = np.loadtxt("data/9_11.txt").flatten()
n = len(d)
xb = d.mean()
se = sem(d)
a = 0.05
ta = t.ppf(1 - a / 2, n - 1)
L = [xb - se * ta, xb + se * ta]
print(np.round(L, 4))

## 7.12  已知方差的双侧 Z 检验

样本标准差未知但总体标准差 $\sigma=4.2$ 已知，用 `ztest` 检验均值是否为 26；将返回的 z 统计量记为 `tstat`（避免覆盖 t 分布），再结合样本标准差换算，并与双侧 $\alpha/2$ 分位数比较（源程序 ex9_12.py）。

In [ ]:
alpha = 0.05
sigma = 4.2
a = np.array([26.01, 26.00, 25.98, 25.86, 26.32, 25.58, 25.32, 25.89, 26.32, 26.18])
tstat, p = ztest(a, value=26)
z = tstat * a.std(ddof=1) / sigma
za = norm.ppf(1 - alpha / 2)
print("z 统计量：", z)
print("p 值：", p)
print("分位数：", za)

## 7.13  单样本右尾 Z 检验

读取 `data/9_13.txt` 展平，用 `ztest` 做大样本右尾（`alternative="larger"`）检验 $H_0:\mu=225$，打印检验统计量（源程序 ex9_13.py）。

In [ ]:
a = np.loadtxt("data/9_13.txt").flatten()
ts, _ = ztest(a, value=225, alternative="larger")
print("t 统计量值：", ts)

## 7.14  两独立样本 t 检验（库函数 + 手写）

读取 `data/9_14.txt` 两行数据为 a、b，先用 `ttest_ind` 做右尾两样本 t 检验，打印统计量、p 值与自由度；随后不依赖库函数，按合并方差公式$t=\dfrac{\bar{x}_a-\bar{x}_b}{s_p\sqrt{1/n_1+1/n_2}}$ 手写复现同一统计量（源程序 ex9_14.py）。

In [ ]:
with open("data/9_14.txt") as fh:
    lines = fh.readlines()
a = np.array(lines[0].split(), dtype=float)
b = np.array(lines[1].split(), dtype=float)
tstat, p, dof = ttest_ind(a, b, alternative="larger")
print("检验统计量为：", tstat)
print("p 值为:", p)
print("自由度为：", dof)

n1, n2 = len(a), len(b)
xa, xb = a.mean(), b.mean()
sa2, sb2 = a.var(ddof=1), b.var(ddof=1)
sp = np.sqrt(((n1 - 1) * sa2 + (n2 - 1) * sb2) / (n1 + n2 - 2))
ts = (xa - xb) / (sp * np.sqrt(1 / n1 + 1 / n2))
print("检验统计量为：", ts)

## 7.15  泊松分布的卡方拟合优度检验

给定观测频数（次数 0~6），以样本均值估计泊松参数 $\lambda$，计算 0、1、2 及“$\ge 3$”合并组的期望频数与观测频数；分别用 `chisquare`（`ddof=1`）和$\sum O^2/E-n$ 两种公式计算统计量并比较（源程序 ex9_15.py）。

In [ ]:
obs = np.array([36, 40, 19, 2, 0, 2, 1])
vals = np.arange(7)
n = obs.sum()
lamda = obs @ vals / n
p0 = ss.poisson.pmf(range(3), lamda)
p = np.hstack([p0, 1 - p0.sum()])
ex = n * p
ob = np.hstack([obs[:3], obs[3:].sum()])
kf1 = ss.chisquare(ob, ex, ddof=1).statistic
kf2 = (ob**2 / ex).sum() - n
print(kf1)
print(kf2)

## 7.16  指数分布的卡方拟合优度检验

以 100、200、300、$\infty$ 为分点，用 Exponential(scale=200) 的 CDF 求各区间概率，与四段观测频数比较；用 `chisquare` 计算统计量与 p 值，输出统计量（保留 4 位，源程序 ex9_16.py）。

In [ ]:
edges = np.array([100, 200, 300, np.inf])
cdf_vals = ss.expon.cdf(edges, scale=200)
p = np.hstack([cdf_vals[0], np.diff(cdf_vals)])
ob = np.array([121, 78, 43, 58])
n = ob.sum()
ex = n * p
kf, pv = ss.chisquare(ob, ex)
print("检验统计量的值：", round(kf, 4))

## 7.17  K-S 正态性检验

读取 `data/9_17.txt` 所有数值，以样本均值、样本标准差为参数，用 `kstest` 做正态性Kolmogorov–Smirnov 检验，打印统计量与 p 值（各保留 4 位，源程序 ex9_17.py）。

In [ ]:
with open("data/9_17.txt") as fh:
    d = fh.readlines()
b = np.array([float(e) for line in d for e in line.split()])
xb, s = b.mean(), b.std(ddof=1)
st, p = ss.kstest(b, "norm", (xb, s))
print("统计量：", round(st, 4))
print("p 值", round(p, 4))

## 7.18  单因素方差分析（含缺失值）

读取 `data/9_18.xlsx`，转置后剔除 NaN 得到观测值 y，并按组构造因子 x（4 组、各组样本数分别为 5、4、4、3）；用 `ols("y~C(x)")` 建模后 `anova_lm` 输出方差分析表（源程序 ex9_18.py）。

In [ ]:
a = pd.read_excel("data/9_18.xlsx", header=None)
b = a.values.T
y = b[~np.isnan(b)]
x = np.hstack([np.ones(5), np.full(4, 2), np.full(4, 3), np.full(3, 4)])
data = {"x": x, "y": y}
model = sm.formula.ols("y~C(x)", data).fit()
anovat = sm.stats.anova_lm(model)
print(anovat)

## 7.19  单因素方差分析（F 临界值与总偏差平方和）

读取 `data/9_19.txt`，展平观测值并按处理（4 行）与水平（1~5）构造因子 x（`np.repeat` 即每水平重复 4 次）；`ols`+`anova_lm` 得到方差表后，用 F 分布求$\alpha=0.05$ 上侧临界值，并给出总的偏差平方和（源程序 ex9_19.py）。

In [ ]:
arr = np.loadtxt("data/9_19.txt")
y = arr.flatten()
x = np.repeat(np.arange(1, 6), 4)
data = {"x": x, "y": y}
m = sm.formula.ols("y~C(x)", data).fit()
s = sm.stats.anova_lm(m)
fa = f.ppf(0.95, s.df[0], s.df[1])
ts = s.sum_sq.sum()
print(s)
print("临界值：", round(fa, 4))
print("总的偏差平方和：", round(ts, 4))

## 7.20  双因素无交互方差分析

读取 `data/9_20.txt`（4×3 矩阵），行因子 x1 取 1~4（每个重复 3 次）、列因子 x2 取 1~3（循环 4 次）；用可加模型 `y~C(x1)+C(x2)` 做双因素方差分析并输出方差表（源程序 ex9_20.py）。

In [ ]:
a = np.loadtxt("data/9_20.txt")
x1 = np.repeat(np.arange(1, 5), 3)
x2 = np.tile(np.arange(1, 4), 4)
data = {"x1": x1, "x2": x2, "y": a.flatten()}
m = sm.formula.ols("y~C(x1)+C(x2)", data).fit()
s = sm.stats.anova_lm(m)
print(s)

## 7.21  双因素含交互方差分析

读取 `data/9_21.txt`（4×6 矩阵），行因子 x1 取 1~4（每个重复 6 次）、列因子 x2 按 [1,1,2,2,3,3] 循环 4 次；用带交互项的模型 `y~C(x1)*C(x2)` 做双因素方差分析并输出方差表（源程序 ex9_21.py）。

In [ ]:
a = np.loadtxt("data/9_21.txt")
x1 = np.repeat(np.arange(1, 5), 6)
x2 = np.tile([1, 1, 2, 2, 3, 3], 4)
data = {"x1": x1, "x2": x2, "y": a.flatten()}
m = sm.formula.ols("y~C(x1)*C(x2)", data).fit()
s = sm.stats.anova_lm(m)
print(s)